In [ ]:
%pip uninstall -y -q transformers huggingface-hub datasets pyarrow accelerate
%pip install -q --no-cache-dir \
  "transformers==4.57.1" \
  "huggingface-hub==0.36.0" \
  "datasets==2.21.0" \
  "pyarrow==17.0.0" \
  "accelerate>=0.30,<1.0"

!git clone https://github.com/phisomni-edu/cs4120-project /content/cs4120-project
%cd /content/cs4120-project

import sys
from pathlib import Path
import torch

repo_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path('/content/cs4120-project'),
]
repo_root = None
for candidate in repo_candidates:
    if (candidate / 'src').exists() and (candidate / 'data').exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError('Could not locate repo root with src/ and data/ directories.')

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

DATA_DIR = repo_root / 'data'
RESULTS_DIR = repo_root / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'PyTorch: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Repo root:', repo_root)
print('Data dir:', DATA_DIR)
print('Results dir:', RESULTS_DIR)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 277.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 387.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 363.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 339.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.
Cloning into '/content/cs4120-project'...
remote: Enumerating objects: 345, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (90

In [ ]:
import ast
import gc
import time

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import MultiLabelBinarizer

# shared experiment grid (fractions x seeds)
DATA_FRACTIONS = [0.01, 0.05, 0.10, 0.25, 0.50, 1.0]
SEEDS = [42, 7, 21]

FRACTION_TAG = {
    0.01: '1pct',
    0.05: '5pct',
    0.10: '10pct',
    0.25: '25pct',
    0.50: '50pct',
    1.00: '100pct',
}

def parse_list_cell(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    try:
        out = ast.literal_eval(x)
        return out if isinstance(out, list) else []
    except Exception:
        return []

def choose_text_column(df):
    for col in ['text_clean_transformer', 'text']:
        if col in df.columns:
            return col
    raise ValueError('No supported text column found. Expected text_clean_transformer or text.')

# base split files generated by 00_eda
TRAIN_FULL_PATH = DATA_DIR / 'train.csv'
VAL_PATH = DATA_DIR / 'val.csv'
TEST_PATH = DATA_DIR / 'test.csv'

if not TRAIN_FULL_PATH.exists() or not VAL_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError('Missing one of train.csv / val.csv / test.csv in data/. Run 00_eda.ipynb export first.')

train_full_df = pd.read_csv(TRAIN_FULL_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

text_col = choose_text_column(train_full_df)
label_col = 'label_names' if 'label_names' in train_full_df.columns else 'labels'

for df in [train_full_df, val_df, test_df]:
    df['labels_list'] = df[label_col].apply(parse_list_cell)
    if text_col in df.columns:
        df['text_model'] = df[text_col].fillna(df['text']).astype(str)
    else:
        df['text_model'] = df['text'].astype(str)

label_classes = sorted({lbl for labels in train_full_df['labels_list'] for lbl in labels})
if not label_classes:
    raise ValueError('No labels found after parsing. Check data/*.csv label formatting.')

print('Using label column:', label_col)
print('Text column:', text_col)
print('Num labels:', len(label_classes))
print('Sample labels:', label_classes[:8])

def fraction_train_path(data_fraction, seed):
    frac = float(data_fraction)
    seed = int(seed)
    if abs(frac - 1.0) < 1e-12:
        return DATA_DIR / 'train.csv'
    tag = FRACTION_TAG.get(frac)
    if tag is None or tag == '100pct':
        raise ValueError(f'Unsupported fraction: {data_fraction}')
    return DATA_DIR / f'train_{tag}_seed{seed}.csv'

for s in SEEDS:
    for f in DATA_FRACTIONS:
        p = fraction_train_path(f, s)
        if not p.exists():
            raise FileNotFoundError(f'Missing partition file: {p}')

print('All partition files found for configured fractions/seeds.')


Using label column: label_names
Text column: text_clean_transformer
Num labels: 28
Sample labels: ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity']
All partition files found for configured fractions/seeds.


In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed,
)
import inspect
from accelerate import Accelerator
from sklearn.metrics import f1_score
from src.evaluate import evaluate_run

if 'keep_torch_compile' not in inspect.signature(Accelerator.unwrap_model).parameters:
    _orig_unwrap_model = Accelerator.unwrap_model

    def _unwrap_model_compat(self, model, *args, **kwargs):
        kwargs.pop('keep_torch_compile', None)
        return _orig_unwrap_model(self, model, *args, **kwargs)

    Accelerator.unwrap_model = _unwrap_model_compat

MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 128
DEFAULT_PARAMS = {
    'learning_rate': 2e-5,
    'batch_size': 16,
    'num_epochs': 3,
    'weight_decay': 0.01,
}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
mlb = MultiLabelBinarizer(classes=label_classes)
mlb.fit([label_classes])

def prepare_model_df(df):
    out = df.copy()
    out['labels_binary'] = list(mlb.transform(out['labels_list']))
    return out

VAL_MODEL_DF = prepare_model_df(val_df)
TEST_MODEL_DF = prepare_model_df(test_df)

class EmotionDataset(Dataset):
    def __init__(self, df):
        self.encodings = tokenizer(
            df['text_model'].tolist(),
            truncation=True,
            padding='max_length',
            max_length=MAX_LENGTH,
            return_tensors='pt',
        )
        self.labels = torch.tensor(np.stack(df['labels_binary']), dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx],
        }

def _compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds = (probs >= 0.5).astype(int)
    return {
        'f1_micro': f1_score(labels.astype(int), preds, average='micro', zero_division=0),
        'f1_macro': f1_score(labels.astype(int), preds, average='macro', zero_division=0),
    }

def load_train_fraction_df(data_fraction, seed):
    p = fraction_train_path(data_fraction, seed)
    df = pd.read_csv(p)
    df['labels_list'] = df[label_col].apply(parse_list_cell)
    if text_col in df.columns:
        df['text_model'] = df[text_col].fillna(df['text']).astype(str)
    else:
        df['text_model'] = df['text'].astype(str)
    return prepare_model_df(df)

def run_single_distilbert_experiment(data_fraction, seed, params=None):
    cfg = dict(DEFAULT_PARAMS)
    if params is not None:
        cfg.update(params)

    seed = int(seed)
    data_fraction = float(data_fraction)
    set_seed(seed)
    t0 = time.time()

    train_model_df = load_train_fraction_df(data_fraction, seed)
    train_ds = EmotionDataset(train_model_df)
    val_ds = EmotionDataset(VAL_MODEL_DF)
    test_ds = EmotionDataset(TEST_MODEL_DF)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(label_classes),
        problem_type='multi_label_classification',
    )

    # trainer config used for all distilbert runs
    args = TrainingArguments(
        output_dir=f'/tmp/distilbert_frac{int(data_fraction*100)}_seed{seed}',
        num_train_epochs=cfg['num_epochs'],
        per_device_train_batch_size=cfg['batch_size'],
        per_device_eval_batch_size=max(1, cfg['batch_size'] * 2),
        learning_rate=cfg['learning_rate'],
        weight_decay=cfg['weight_decay'],
        eval_strategy='epoch',
        save_strategy='no',
        logging_strategy='steps',
        logging_steps=50,
        report_to=[],
        seed=seed,
        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=_compute_metrics,
    )

    trainer.train()
    pred_out = trainer.predict(test_ds)
    logits_np = pred_out.predictions
    y_true = pred_out.label_ids.astype(int)
    y_pred = (torch.sigmoid(torch.tensor(logits_np)) >= 0.5).int().numpy()

    eval_out = evaluate_run(
        method='distilbert',
        data_fraction=data_fraction,
        seed=seed,
        label_names=[str(x) for x in label_classes],
        y_true=y_true,
        y_pred=y_pred,
    )

    elapsed_min = (time.time() - t0) / 60.0
    eval_out['overall']['train_rows'] = len(train_model_df)
    eval_out['overall']['learning_rate'] = cfg['learning_rate']
    eval_out['overall']['batch_size'] = cfg['batch_size']
    eval_out['overall']['num_epochs'] = cfg['num_epochs']
    eval_out['overall']['weight_decay'] = cfg['weight_decay']
    eval_out['overall']['elapsed_min'] = elapsed_min

    del trainer, model, train_ds, val_ds, test_ds, pred_out
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return eval_out


In [ ]:
# resumable csv outputs
OVERALL_PATH = RESULTS_DIR / 'distilbert_overall.csv'
PER_CLASS_PATH = RESULTS_DIR / 'distilbert_per_class.csv'
README_PATH = RESULTS_DIR / 'distilbert_results.csv'
ERRORS_PATH = RESULTS_DIR / 'distilbert_errors.csv'

def _safe_read_csv(path):
    if not path.exists():
        return pd.DataFrame()
    try:
        if path.stat().st_size == 0:
            return pd.DataFrame()
        return pd.read_csv(path)
    except (pd.errors.EmptyDataError, FileNotFoundError):
        return pd.DataFrame()

overall_df = _safe_read_csv(OVERALL_PATH)
per_class_df = _safe_read_csv(PER_CLASS_PATH)
errors_df = _safe_read_csv(ERRORS_PATH)

def _dedup_overall(df):
    if df.empty:
        return df
    return (
        df.sort_values(['method', 'seed', 'data_fraction'])
          .drop_duplicates(subset=['method', 'seed', 'data_fraction'], keep='last')
          .reset_index(drop=True)
    )

def _dedup_per_class(df):
    if df.empty:
        return df
    return (
        df.sort_values(['method', 'seed', 'data_fraction', 'emotion'])
          .drop_duplicates(subset=['method', 'seed', 'data_fraction', 'emotion'], keep='last')
          .reset_index(drop=True)
    )

def _persist_results():
    global overall_df, per_class_df, errors_df

    overall_df = _dedup_overall(overall_df)
    per_class_df = _dedup_per_class(per_class_df)

    overall_df.to_csv(OVERALL_PATH, index=False)
    per_class_df.to_csv(PER_CLASS_PATH, index=False)

    if not per_class_df.empty:
        readme_df = per_class_df[['method', 'data_fraction', 'seed', 'emotion', 'f1', 'precision', 'recall']].copy()
        readme_df.to_csv(README_PATH, index=False)

    if not errors_df.empty:
        errors_df.to_csv(ERRORS_PATH, index=False)

def _completed_pairs():
    if overall_df.empty:
        return set()
    return set((float(r.data_fraction), int(r.seed)) for r in overall_df[['data_fraction', 'seed']].itertuples(index=False))

def show_distilbert_progress():
    done = _completed_pairs()
    expected = [(float(f), int(s)) for s in SEEDS for f in DATA_FRACTIONS]
    pending = [p for p in expected if p not in done]

    print(f'Completed runs: {len(done)} / {len(expected)}')
    if pending:
        print('Pending:', pending)
    else:
        print('No pending runs.')

    if not overall_df.empty:
        display(overall_df.sort_values(['seed', 'data_fraction']).tail(10))

def run_and_persist_one(data_fraction, seed, *, force=False, params=None):
    global overall_df, per_class_df, errors_df

    key = (float(data_fraction), int(seed))
    if (not force) and key in _completed_pairs():
        print(f'Skipping completed run: {key}')
        return

    print(f'Running DistilBERT: fraction={data_fraction}, seed={seed}')
    try:
        eval_out = run_single_distilbert_experiment(data_fraction, seed, params=params)

        if not overall_df.empty:
            overall_df = overall_df[~((overall_df['data_fraction'].astype(float) == float(data_fraction)) & (overall_df['seed'].astype(int) == int(seed)))]
        if not per_class_df.empty:
            per_class_df = per_class_df[~((per_class_df['data_fraction'].astype(float) == float(data_fraction)) & (per_class_df['seed'].astype(int) == int(seed)))]

        overall_df = pd.concat([overall_df, eval_out['overall']], ignore_index=True) if not overall_df.empty else eval_out['overall'].copy()
        per_class_df = pd.concat([per_class_df, eval_out['per_class']], ignore_index=True) if not per_class_df.empty else eval_out['per_class'].copy()

        _persist_results()
        macro_f1 = float(eval_out['overall'].iloc[0]['macro_f1'])
        print(f'Completed and saved: macro_f1={macro_f1:.4f}')

    except Exception as exc:
        err_row = pd.DataFrame([{'seed': int(seed), 'data_fraction': float(data_fraction), 'error': str(exc)}])
        errors_df = pd.concat([errors_df, err_row], ignore_index=True) if not errors_df.empty else err_row
        _persist_results()
        print(f'FAILED: {exc}')

# resume only pending runs unless force=True
def run_pending(*, fractions=None, seeds=None, max_runs=None, force=False, params=None):
    fracs = [float(f) for f in (fractions if fractions is not None else DATA_FRACTIONS)]
    seed_list = [int(s) for s in (seeds if seeds is not None else SEEDS)]

    planned = [(f, s) for s in seed_list for f in fracs]
    pending = [p for p in planned if force or p not in _completed_pairs()]

    if max_runs is not None:
        pending = pending[: int(max_runs)]

    print(f'Pending selected runs: {len(pending)}')
    for f, s in pending:
        run_and_persist_one(f, s, force=force, params=params)

    show_distilbert_progress()

show_distilbert_progress()


Completed runs: 0 / 18
Pending: [(0.01, 42), (0.05, 42), (0.1, 42), (0.25, 42), (0.5, 42), (1.0, 42), (0.01, 7), (0.05, 7), (0.1, 7), (0.25, 7), (0.5, 7), (1.0, 7), (0.01, 21), (0.05, 21), (0.1, 21), (0.25, 21), (0.5, 21), (1.0, 21)]


In [ ]:
# run_and_persist_one(0.01, 42)

# run_pending(fractions=[0.10])

# run_pending(max_runs=2)

run_pending()

show_distilbert_progress()


Pending selected runs: 18
Running DistilBERT: fraction=0.01, seed=42


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,No log,0.466692,0.000000,0.000000
2,0.512500,0.379002,0.000000,0.000000
3,0.512500,0.356755,0.000000,0.000000


Completed and saved: macro_f1=0.0000
Running DistilBERT: fraction=0.05, seed=42


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


...

Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.086900,0.089673,0.519124,0.322067
2,0.084800,0.084985,0.553462,0.404442
3,0.075000,0.083793,0.578821,0.417300


Completed and saved: macro_f1=0.4188
Completed runs: 18 / 18
No pending runs.


,method,data_fraction,seed,accuracy,macro_f1,micro_f1,hamming_loss,train_rows,learning_rate,batch_size,num_epochs,weight_decay,elapsed_min
8,distilbert,0.10,21,0.135618,0.018795,0.212117,0.039452,4341,0.00002,16,3,0.01,0.445425
9,distilbert,0.25,21,0.311406,0.155769,0.448640,0.032674,10852,0.00002,16,3,0.01,0.924210
10,distilbert,0.50,21,0.409987,0.321937,0.537299,0.030614,21705,0.00002,16,3,0.01,1.745654
11,distilbert,1.00,21,0.462134,0.418833,0.579721,0.029541,43410,0.00002,16,3,0.01,3.386021
12,distilbert,0.01,42,0.000000,0.000000,0.000000,0.041650,434,0.00002,16,3,0.01,0.164254
13,distilbert,0.05,42,0.000000,0.000000,0.000000,0.041650,2170,0.00002,16,3,0.01,0.276519
14,distilbert,0.10,42,0.119956,0.017520,0.191601,0.039650,4341,0.00002,16,3,0.01,0.435875
15,distilbert,0.25,42,0.311959,0.161755,0.446246,0.032812,10852,0.00002,16,3,0.01,0.918144
16,distilbert,0.50,42,0.407776,0.301504,0.533586,0.030798,21705,0.00002,16,3,0.01,1.713978
17,distilbert,1.00,42,0.457527,0.416215,0.576376,0.029620,43410,0.00002,16,3,0.01,3.354454


Completed runs: 18 / 18
No pending runs.


,method,data_fraction,seed,accuracy,macro_f1,micro_f1,hamming_loss,train_rows,learning_rate,batch_size,num_epochs,weight_decay,elapsed_min
8,distilbert,0.10,21,0.135618,0.018795,0.212117,0.039452,4341,0.00002,16,3,0.01,0.445425
9,distilbert,0.25,21,0.311406,0.155769,0.448640,0.032674,10852,0.00002,16,3,0.01,0.924210
10,distilbert,0.50,21,0.409987,0.321937,0.537299,0.030614,21705,0.00002,16,3,0.01,1.745654
11,distilbert,1.00,21,0.462134,0.418833,0.579721,0.029541,43410,0.00002,16,3,0.01,3.386021
12,distilbert,0.01,42,0.000000,0.000000,0.000000,0.041650,434,0.00002,16,3,0.01,0.164254
13,distilbert,0.05,42,0.000000,0.000000,0.000000,0.041650,2170,0.00002,16,3,0.01,0.276519
14,distilbert,0.10,42,0.119956,0.017520,0.191601,0.039650,4341,0.00002,16,3,0.01,0.435875
15,distilbert,0.25,42,0.311959,0.161755,0.446246,0.032812,10852,0.00002,16,3,0.01,0.918144
16,distilbert,0.50,42,0.407776,0.301504,0.533586,0.030798,21705,0.00002,16,3,0.01,1.713978
17,distilbert,1.00,42,0.457527,0.416215,0.576376,0.029620,43410,0.00002,16,3,0.01,3.354454


In [10]:
# Archived troubleshooting cell.
# No action needed: environment pinning is handled in the first setup cell.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 54.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.11.0 which is incompatible.
cudf-cu12 26.2.1 requires cuda-toolkit[nvcc,nvrtc]==12.*, but you have cuda-toolkit 13.0.2 

In [16]:
# Archived troubleshooting cell.
# Compatibility checks were consolidated into the main training setup cell.


torch 2.10.0+cu128
transformers 4.57.1
accelerate 0.34.2
datasets 2.21.0
hub 0.36.0
fsspec 2024.6.1
pyarrow 17.0.0
unwrap_model: (self, model, keep_fp32_wrapper: 'bool' = True)


In [17]:
# Archived troubleshooting cell.
# The unwrap-model compatibility patch now runs in the main DistilBERT setup cell.


Applied accelerate unwrap_model compatibility patch.
